# JOB POSTING SCRAPER





This notebook is mainly used for scraping job postings in LinkedIn, specifically targeting data science/data scientist roles posted in the Philippines. Initially tested using Google COLAB so results may differ when notebook is ran again for scraping a data.  

### Initial setup for scraping data

In [ ]:
import requests
import time
import random
from bs4 import BeautifulSoup
from langchain_core.documents import Document

def fetch_linkedin_jobs(keywords, location="Philippines", limit=None, delay=2.0, randomize=True):

    """
    Fetch LinkedIn job postings with pagination, a result limit, and a delay between requests.

    :param keywords: Job search keywords.
    :param location: Location filter.
    :param limit: Maximum number of job documents to return. None = all available.
    :param delay: Base delay in seconds between page requests.
    :param randomize: If True, delay is random between delay*0.5 and delay*1.5.
    :return: List of Document objects.
    """

    base_url = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    docs = []
    start = 0
    page_size = 25

    while True:
        params = {
            "keywords": keywords,
            "location": location,
            "start": start
        }
        response = requests.get(base_url, params=params, headers=headers)

        if response.status_code != 200:
            print(f"Request failed with status {response.status_code}. Stopping.")
            break

        # Parse the HTML content to extract job cards
        soup = BeautifulSoup(response.content, "html.parser")
        cards = soup.find_all("div", class_="base-card")

        if not cards:
            print("No more job cards found. Stopping.")
            break

        for card in cards:
            title = card.find("h3", class_="base-search-card__title")
            company = card.find("h4", class_="base-search-card__subtitle")
            link = card.find("a", class_="base-card__full-link")

            content = (
                f"Title: {title.text.strip() if title else 'N/A'}\n"
                f"Company: {company.text.strip() if company else 'N/A'}\n"
                f"Link: {link['href'] if link else 'N/A'}"
            )
            docs.append(Document(page_content=content))

            if limit is not None and len(docs) >= limit:
                return docs[:limit]

        if randomize:
            # Random delay between 0.5*delay and 1.5*delay
            actual_delay = random.uniform(delay * 0.5, delay * 1.5)
        else:
            actual_delay = delay

        print(f"Waiting {actual_delay:.2f} seconds before next page...")
        time.sleep(actual_delay)

        # Move to next page
        start += page_size

        if limit is not None and len(docs) >= limit:
            return docs[:limit]

    return docs

In [ ]:
# Test the function if it works
jobs = fetch_linkedin_jobs("Data Scientist", limit=100, delay=10.0, randomize=False)

for doc in jobs:
    print(doc.page_content)
    print("---")

Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Title: Data Scientist
Company: Maya
Link: https://ph.linkedin.com/jobs/view/data-scientist-at-maya-4445458453?position=1&pageNum=0&refId=%2BHwhkctx6m3ojqP8qs3uJA%3D%3D&trackingId=QtjzPcgspnCQD7KYf3d7Ww%3D%3D
---
Title: Applied Data Scientist
Company: Emapta Global
Link: https://ph.linkedin.com/jobs/view/applied-data-scientist-at-emapta-global-4445655406?position=2&pageNum=0&refId=%2BHwhkctx6m3ojqP8qs3uJA%3D%3D&trackingId=DQI04T8chn3JbP%2FjHFsFdA%3D%3D
---
Title: Data Scientist
Company: Afni, Inc.
Link: https://ph.linkedin.com/jobs/view/data-scientist-at-afni-inc-4443471809?

In [ ]:
import re

def extract_requirements_section(text):
    """
    Given full job description text, isolate the requirements/qualifications section.
    """
    if text == "N/A":
        return "N/A"

    # starting portion for the requirements section in linkedin postings
    start_keywords = [
    "requirements", "qualifications", "basic qualifications",
    "minimum qualifications", "preferred qualifications", "essential qualifications",
    "minimum requirements", "key requirements", "role requirements",
    "skills", "skills & experience", "skills and experience",
    "education & experience", "prerequisites",

    "what you'll need", "what you need", "what you bring", "what you'll bring",
    "what you need to succeed", "what makes you a great fit",
    "what we're looking for", "what we are looking for", "who we're looking for",
    "who you are", "about you", "your background", "your profile",
    "you should have", "you have", "ideal candidate", "the ideal candidate", "looking for"

    "preferred", "good to have", "nice to have", "nice-to-haves",
    "must haves", "must-haves", "looking for"
    ]

    end_keywords = [
        "benefits", "what we offer", "about us", "about the company",
        "how to apply", "compensation", "perks", "equal opportunity"
    ]

    lines = text.split("\n")
    lower_lines = [l.lower().strip() for l in lines]

    # Find the starting index of the requirements section
    start_idx = None
    for i, line in enumerate(lower_lines):
        if any(kw in line for kw in start_keywords) and len(line) < 60:
            start_idx = i
            break

    if start_idx is None:
        return "N/A"

    end_idx = len(lines)
    for i in range(start_idx + 1, len(lower_lines)):
        if any(kw in lower_lines[i] for kw in end_keywords) and len(lower_lines[i]) < 60:
            end_idx = i
            break

    return "\n".join(lines[start_idx:end_idx]).strip()

In [ ]:
import re

def fetch_job_requirements(link, headers, delay=1.5, randomize=True):
    """
    Given a LinkedIn job link, extract the job ID and fetch the stated requirements or qualifications.
    :param link: url
    :param headers: HTTP headers for the request.
    :param delay: Base delay in seconds between page requests.
    :param randomize: If True, delay is random between delay*0.5 and delay*1.5.
    :return: the requirements or qualifications section
    """
    match = re.search(r"-(\d+)(?:\?|$)", link)
    if not match:
        return "N/A"

    # Construct the API URL to fetch job details
    job_id = match.group(1)
    detail_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"

    response = requests.get(detail_url, headers=headers)
    if response.status_code != 200:
        return "N/A"

    # Parse the job description HTML to extract the requirements section
    soup = BeautifulSoup(response.content, "html.parser")
    desc_div = soup.find("div", class_="show-more-less-html__markup")
    requirements = desc_div.get_text(separator="\n", strip=True) if desc_div else "N/A"
    requirements = extract_requirements_section(requirements)

    actual_delay = random.uniform(delay * 0.5, delay * 1.5) if randomize else delay
    time.sleep(actual_delay)

    return requirements

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Fetch jobs
jobs = fetch_linkedin_jobs("Data Scientist", limit=100, delay=10.0, randomize=False)

# Fetch requirements for each job and append to page_content
for doc in jobs:
    match = re.search(r"Link:\s*(\S+)", doc.page_content)
    link = match.group(1) if match else None

    requirements = fetch_job_requirements(link, HEADERS, delay=10.0, randomize=True)
    doc.page_content += f"\nRequirements: {requirements}"

    print(doc.page_content)
    print("---")

Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Waiting 10.00 seconds before next page...
Title: Data Scientist
Company: Maya
Link: https://ph.linkedin.com/jobs/view/data-scientist-at-maya-4445458453?position=1&pageNum=0&refId=Iwf7LgvZY%2FdnxjdgmEKQHg%3D%3D&trackingId=6TkF0UK1%2F4%2FnARrFNfkq1A%3D%3D
Requirements: We are looking for a
Data Scientist
to build and scale Maya’s intelligence layer. You will be responsible for the end-to-end development of production-ready machine learning solutions designed to address intricate business problems. By aligning deeply with cross-functional requirements, you will ensure our data assets are fully leveraged to achieve agg

### Saving the scraped data

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json
import csv

def parse_doc(doc):
    """Convert a Document's page_content into a dict."""
    data = {"title": "N/A", "company": "N/A", "link": "N/A", "requirements": "N/A"}
    labels = {"Title:": "title", "Company:": "company", "Link:": "link", "Requirements:": "requirements"}

    current_key = None

    # Parse each line of the document content
    for line in doc.page_content.split("\n"):
        matched_label = next((lbl for lbl in labels if line.startswith(lbl)), None)
        if matched_label:
            current_key = labels[matched_label]
            data[current_key] = line.replace(matched_label, "").strip()
        elif current_key:
            # Continuation of a multi-line field (e.g. requirements)
            data[current_key] += "\n" + line.strip()

    return data


In [ ]:

records = [parse_doc(doc) for doc in jobs]
# "/content/drive/MyDrive/COLAB/for_scraping/linkedin_jobs.csv"
csv_path = "/job_postings_ph/linkedin_jobs.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=["title", "company", "link", "requirements"]
    )
    writer.writeheader()
    writer.writerows(records)